In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## label encoding
from sklearn.preprocessing import LabelEncoder

In [ ]:
data = pd.read_csv('IPL_Full_Merged_Dataset_2008_2020.csv')
data.iloc[0]

In [ ]:
# data["dismissal_kind"] = data["dismissal_kind"].fillna("not out")
# dismissal_kind_enc = LabelEncoder()
# dismissal_kind_enc.fit(data['dismissal_kind'])
# data['dismissal_kind'] = dismissal_kind_enc.transform(data['dismissal_kind'])


# data["player_dismissed"] = data["player_dismissed"].fillna("not dismissed")
# player_dismissed_enc = LabelEncoder()
# player_dismissed_enc.fit(data['player_dismissed'])
# data['player_dismissed'] = player_dismissed_enc.transform(data['player_dismissed'])


# data["fielder"] = data["fielder"].fillna("no fielder")
# fielder_enc = LabelEncoder()
# fielder_enc.fit(data['fielder'])
# data['fielder'] = fielder_enc.transform(data['fielder'])


# data["extras_type"] = data["extras_type"].fillna("no extras")
# extras_type_enc = LabelEncoder()
# extras_type_enc.fit(data['extras_type'])
# data['extras_type'] = extras_type_enc.transform(data['extras_type'])


# data["eliminator"] = data["eliminator"].fillna("None")
# eliminator_enc = LabelEncoder()
# eliminator_enc.fit(data['eliminator'])
# data['eliminator'] = eliminator_enc.transform(data['eliminator'])


# data["method"] = data["method"].fillna("No Method")
# method_enc = LabelEncoder()
# method_enc.fit(data['method'])
# data['method'] = method_enc.transform(data['method'])



In [ ]:
data.drop(["dismissal_kind", "player_dismissed", "extras_type", "eliminator", "method", 'neutral_venue', 'umpire1', 'umpire2', 'date', 'player_of_match','team1', 'team2', 'result','result_margin','fielder' ],axis=1, inplace=True)

In [ ]:
data = data.replace('Rising Pune Supergiant', 'Rising Pune Supergiants')
data['winner'] = data['winner'].fillna('None')
data['bowling_team'] = data['bowling_team'].fillna('None')

team_enc = LabelEncoder()

list_of_teams = [
    'Delhi Daredevils',
    'Rising Pune Supergiants',
    'Gujarat Lions',
    'Delhi Capitals',
    'Royal Challengers Bangalore',
    'Mumbai Indians',
    'Chennai Super Kings',
    'Kings XI Punjab',
    'Kochi Tuskers Kerala',
    'Kolkata Knight Riders',
    'Sunrisers Hyderabad',
    'Deccan Chargers',
    'Rajasthan Royals',
    'Pune Warriors',
    'None',
    ]

team_enc.fit(list_of_teams)


# data['team1'] = team_enc.transform(data['team1'])
# data['team2'] = team_enc.transform(data['team2'])
data['toss_winner'] = team_enc.transform(data['toss_winner'])
data['winner'] = team_enc.transform(data['winner'])
data['batting_team'] = team_enc.transform(data['batting_team'])
data['bowling_team'] = team_enc.transform(data['bowling_team'])




In [ ]:
# fill all nan values

for i in data.columns:
    if type(data[i].iloc[0]) == str:
        data[i] = data[i].fillna('None')
    else:
        data[i] = data[i].fillna(0)

In [ ]:
venue_enc = LabelEncoder()
venue_enc.fit(data['venue'])
data['venue'] = venue_enc.transform(data['venue'])

city_enc = LabelEncoder()
city_enc.fit(data['city'])
data['city'] = city_enc.transform(data['city'])

# umpire1_enc = LabelEncoder()
# umpire1_enc.fit(data['umpire1'])
# data['umpire1'] = umpire1_enc.transform(data['umpire1'])

# umpire2_enc = LabelEncoder()
# umpire2_enc.fit(data['umpire2'])
# data['umpire2'] = umpire2_enc.transform(data['umpire2'])

toss_decision_enc = LabelEncoder()
toss_decision_enc.fit(data['toss_decision'])
data['toss_decision'] = toss_decision_enc.transform(data['toss_decision'])

In [ ]:
list_of_batsman = list(data["batsman"])
list_of_non_striker = list(data["non_striker"])
list_of_bowler = list(data["bowler"])
all_player_list = list_of_batsman + list_of_non_striker + list_of_bowler

set_of_players = list(set(all_player_list))


players_enc = LabelEncoder()
players_enc.fit(set_of_players)

data['bowler'] = players_enc.transform(data['bowler'])
data['non_striker'] = players_enc.transform(data['non_striker'])
data['batsman'] = players_enc.transform(data['batsman'])
# data['player_of_match'].fillna('None', inplace=True)
# data['player_of_match'] = players_enc.transform(data['player_of_match'])

In [ ]:
# result_enc = LabelEncoder()
# result_enc.fit(data['result'])
# data['result'] = result_enc.transform(data['result'])

In [ ]:
data.iloc[0]    

In [ ]:
## sort data by unique_id, batting_team

data = data.sort_values(by=['id', 'batting_team'])

In [ ]:
## make a current score of the match which have unique id, batting_team and run a cumulative sum of the runs

data['current_score'] = data.groupby(['id', 'batting_team'])['total_runs'].cumsum()

In [ ]:
## where inning is 1, make the target score as current score of the match
## where inning is 2, make the target score as the current score of the match where inning is 1 with the same unique id and max current score


# Assuming 'data' is your DataFrame
# Create a new column 'target_score' initialized with NaN
data['target_score'] = 0

# Set 'target_score' for Inning 1 directly to 'current_score'
data.loc[data['inning'] == 1, 'target_score'] = data['current_score']

# For Inning 2, we need to find the maximum 'current_score' from Inning 1 of the same match
# First, get maximum 'current_score' for each 'id' when 'inning' is 1
max_scores = data[data['inning'] == 1].groupby('id')['current_score'].max().reset_index()
max_scores.rename(columns={'current_score': 'max_current_score'}, inplace=True)

# Merge this max score back to the main DataFrame to use it for setting 'target_score' for Inning 2
data = pd.merge(data, max_scores, on='id', how='left')

# Now set 'target_score' for Inning 2
data.loc[data['inning'] == 2, 'target_score'] = data['max_current_score']

# Drop the temporary column 'max_current_score' as it's no longer needed
data.drop(columns=['max_current_score'], inplace=True)

# Now, 'data' DataFrame has the 'target_score' set as required
print(data[['id', 'inning', 'current_score', 'target_score']].head())  # Show some data to verify



In [ ]:
## make a new column of number of balls played in the match

data["balls_played"] = data["over"]*6 + data["ball"]
data["balls_remaining"] = 120 - data["balls_played"]

In [ ]:
data['wickets_taken'] = data.groupby(['id', 'inning'])['is_wicket'].cumsum()
data['wickets_left'] = 10 - data['wickets_taken']

In [ ]:
## drop rows where balls_remaining is less than 1

data = data[data['balls_remaining'] > 0]

In [ ]:
data["current_run_rate"] = data["current_score"]/(data["balls_played"]/6)

data["required_run_rate"] = data["current_run_rate"]

# Now set 'required_run_rate' for Inning 2
data.loc[data['inning'] == 2, 'required_run_rate'] = (data['target_score'] - data['current_score'])/(data['balls_remaining']/6)


In [ ]:
data['required_run_rate']

In [ ]:
## test train split on the basis of id column

# Get unique IDs and shuffle them
unique_ids = data['id'].unique()
np.random.shuffle(unique_ids)

# Calculate split index for 80-20 ratio
split_idx = int(len(unique_ids) * 0.8)

# Split IDs into training and testing
train_ids = unique_ids[:split_idx]
test_ids = unique_ids[split_idx:]

# Create training and testing sets
train_df = data[data['id'].isin(train_ids)]
test_df = data[data['id'].isin(test_ids)]

# Now, train_df and test_df are your training and testing sets respectively
print("Training set size:", train_df.shape)
print("Testing set size:", test_df.shape)


X_train = train_df.drop(['winner'], axis=1).to_numpy()
y_train = train_df['winner'].to_numpy()

X_test = test_df.drop(['winner'], axis=1).to_numpy()
y_test = test_df['winner'].to_numpy()


# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# pca = PCA(n_components=5)
# X_train = pca.fit_transform(X_train)
# X_test = pca.transform(X_test)



In [ ]:
data_frame_columns = data.columns

In [ ]:
del data

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# print(confusion_matrix(y_test,y_pred))
# print(classification_report(y_test,y_pred))
print(accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

# print(confusion_matrix(y_test,y_pred))
# print(classification_report(y_test,y_pred))
print(accuracy_score(y_test, y_pred))

In [ ]:
## drop 'winner' from data_frame_columns list

data_frame_columns = data_frame_columns.drop('winner')

In [ ]:
pd.DataFrame(classifier.feature_importances_,
                                   index = data_frame_columns,
                                   columns=['importance']).sort_values('importance', ascending=False)

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier()
xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)


# print(confusion_matrix(y_test,y_pred))
# print(classification_report(y_test,y_pred))
print(accuracy_score(y_test, y_pred))

In [ ]:
importance = sorted(xgb.feature_importances_, reverse=True)

# Summarize feature importance
for i, v in enumerate(importance):
    print(f'Feature: {X_train.columns[i]}, Score: {v:.5f}')


In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)

y_pred = lgbm.predict(X_test)

# print(confusion_matrix(y_test,y_pred))
# print(classification_report(y_test,y_pred))
print(accuracy_score(y_test, y_pred))


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)

y_pred = gbc.predict(X_test)

# print(confusion_matrix(y_test,y_pred))
# print(classification_report(y_test,y_pred))
print(accuracy_score(y_test, y_pred))

In [ ]:
## train ensemble model voting classifier

from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

estimators = [
    ('rf', RandomForestClassifier(random_state=42)),
    ('xgb', XGBClassifier()),
    ('lgbm', LGBMClassifier()),
    ('gbc', GradientBoostingClassifier()),
]

ensemble = VotingClassifier(estimators, n_jobs=-1, voting='hard')

ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)

print(accuracy_score(y_test, y_pred))

In [ ]:
## train ensemble model voting classifier

from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

estimators = [
    ('rf', RandomForestClassifier(random_state=42)),
    ('xgb', XGBClassifier()),
    ('lgbm', LGBMClassifier()),
    ('gbc', GradientBoostingClassifier()),
]

ensemble = VotingClassifier(estimators, n_jobs=-1, voting='soft')

ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)

print(accuracy_score(y_test, y_pred))